# LAB ASSIGNMENT 5: LSTM-Based AI Agent for Sequence Prediction

## Group Members
- **Mohit Patil** (202301040272)
- **Parimal Ahire** (202301040067)
- **Rajveersinh Kher** (202301040233)
- **Atharva Suryawanshi** (202301040283)

**Course:** Deep Learning Lab
**Task:** LSTM-Based Sequence Prediction (Next Word) with FastAPI Deployment

## Acknowledgement of AI Tool Usage

- **Tool Name:** Antigravity (Powered by Google DeepMind)
- **Purpose:** Code generation, architectural planning, and documentation assistance.
- **Sections:** Used for structuring the notebook, implementing the LSTM architecture, drafting the mathematical explanations, and setting up the FastAPI deployment script.

## 1. LSTM Mathematical Model

Long Short-Term Memory (LSTM) networks are a type of Recurrent Neural Network (RNN) designed to solve the vanishing gradient problem in standard RNNs, allowing them to learn long-term dependencies in sequence data.

### Core Components:

#### A. Gates
LSTMs use "gates" to control the flow of information:
1.  **Forget Gate ($f_t$):** Decides what information to discard from the previous cell state.
    $$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$
2.  **Input Gate ($i_t$):** Decides which new values to update in the cell state.
    $$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$
3.  **Output Gate ($o_t$):** Decides what the next hidden state should be based on the cell state.
    $$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$

#### B. Cell State ($C_t$)
The cell state acts as a long-term memory conveyor belt. It is updated by combining the forgotten old state and new candidate values:
$$\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$$
$$C_t = f_t * C_{2-1} + i_t * \tilde{C}_t$$

#### C. Hidden State ($h_t$)
The hidden state is the output of the cell for the current time step, which is also passed to the next time step:
$$h_t = o_t * \tanh(C_t)$$

### Sequence Learning Significance
LSTM's ability to maintain a persistent cell state makes it highly effective for natural language tasks like next-word prediction, where the context of previous words is crucial for predicting the next one.

## 2. Environment Setup & Imports

We import necessary libraries for data processing, model building, visualization, and API deployment.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

print("TensorFlow Version:", tf.__version__)

## 3. Dataset Collection & Preprocessing

We load a small subset of the Wikipedia dataset for fast training.

In [ ]:
# Path to the dataset
dataset_path = r'c:\Users\mrpat\Downloads\PlaintextWikipedia(SimpleEnglish)_Dataset\AllCombined.txt'

# Load a small subset of data for fast training (500 lines)
with open(dataset_path, 'r', encoding='utf-8') as f:
    data = f.readlines()[:500] 

text = " ".join(data).lower()
print(f"Total characters in subset: {len(text)}")

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1
print(f"Total unique words: {total_words}")

# Create input sequences
input_sequences = []
for line in data:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Create predictors and label
X, y = input_sequences[:,:-1], input_sequences[:,-1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("X shape:", X.shape)
print("y shape:", y.shape)

### 3.1 Data Visualization (Before Training)

We visualize the most frequent words in our dataset to understand the distribution of text data.

In [ ]:
from collections import Counter

# Count word frequencies
words = text.split()
word_counts = Counter(words)
common_words = word_counts.most_common(20)

# Plotting
plt.figure(figsize=(12, 6))
sns.barplot(x=[w[0] for w in common_words], y=[w[1] for w in common_words], palette='viridis')
plt.title('Top 20 Most Frequent Words in Sample Data')
plt.xticks(rotation=45)
plt.show()

## 4. LSTM Model Development

We build a sequential model with an Embedding layer, two LSTM layers, and a Dense output layer with softmax activation.

In [ ]:
model = Sequential([
    Embedding(total_words, 100, input_length=max_sequence_len-1),
    LSTM(150, return_sequences=True),
    Dropout(0.2),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

## 5. Model Training

We train the model for 5 epochs (reduced for fast testing) and visualize the training history.

In [ ]:
history = model.fit(X, y, epochs=5, verbose=1)

# Plotting Training Results
plt.figure(figsize=(12, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Accuracy', color='blue')
plt.title('Model Accuracy over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Loss', color='red')
plt.title('Model Loss over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# Save the model and tokenizer
model.save('lstm_next_word.h5')
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
    
print("Model and Tokenizer saved successfully!")

## 6. Testing & Validation (Inference)

We define a function to predict the next 'n' words based on a seed text.

In [ ]:
def predict_next_words(seed_text, next_words=3):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)
        
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

test_seed = "april is the"
print(f"Seed: {test_seed}")
print(f"Prediction: {predict_next_words(test_seed, 5)}")

## 7. Deployment (FastAPI)

Below is the code for deploying the model as an API. This code is intended to be run as a separate script (`app.py`).

In [ ]:
fastapi_code = """
from fastapi import FastAPI
from pydantic import BaseModel
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle
import numpy as np
import uvicorn

app = FastAPI(title='LSTM Next Word Prediction API')

# Load model and tokenizer
model = tf.keras.models.load_model('lstm_next_word.h5')
with open('tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

max_sequence_len = model.input_shape[1] + 1

class PredictionRequest(BaseModel):
    text: str
    num_words: int = 1

@app.post('/predict')
def predict(request: PredictionRequest):
    seed_text = request.text
    for _ in range(request.num_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)
        
        output_word = ''
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += ' ' + output_word
    
    return {'input': request.text, 'prediction': seed_text}

if __name__ == '__main__':
    uvicorn.run(app, host='0.0.0.0', port=8000)
"""

with open('app.py', 'w') as f:
    f.write(fastapi_code)

print("FastAPI script 'app.py' created. To run, use: uvicorn app:app --reload")

## 8. Conclusion & Agent Significance

This LSTM-based sequence prediction system serves as the core 'brain' for a predictive AI agent. By deploying it via FastAPI, we create an industry-ready service that can be integrated into larger workflows (like n8n or chatbots) to provide real-time suggestions, content completion, and automated text generation. 

The LSTM's ability to handle sequence learning ensures that the agent understands context, making it far more capable than simple statistical models.